# Train and Evaluate a Cross-Generator Run (VM workbook)

**Cross-Generator Generalization in Deepfake Detection (IE7374)**

Run this after `00_setup_and_preprocess.ipynb` has produced the crops cache
(`data/processed/` + `data/manifests/crops.parquet`) on this VM.

It builds the leave-one-manipulation-out splits, trains one detector on 3 of the 4
FF++ methods, tests it on the held-out 4th (plus the SimSwap set when it exists),
and prints the seen-vs-unseen gap. Pick your run with the two variables in step 3.

The eight runs are 2 backbones (EfficientNet, XceptionNet) times 4 held-out methods.
Everyone takes one or two so the eight are shared; each writes a results JSON that
combines into the transfer matrix.

## 1. Confirm the crops cache is present
If this fails, run `00_setup_and_preprocess.ipynb` first.

In [ ]:
import pandas as pd, os
assert os.path.exists("data/manifests/crops.parquet"), "run 00_setup_and_preprocess.ipynb first"
m = pd.read_parquet("data/manifests/crops.parquet")
print("crops:", len(m))
print(m.groupby("method").size())

## 2. Build the leave-one-manipulation-out splits
Writes one identity-disjoint fold per held-out method into `data/splits/`.
For each fold, train+val are 3 methods plus real, test is the held-out method plus real.

In [ ]:
!python data/make_splits.py --manifest data/manifests/crops.parquet --out data/splits
!ls -la data/splits

## 3. Pick your run
Set the backbone and the held-out method. This writes a config for that run.

**ADJUST for your VM:** `EPOCHS`, `BATCH_SIZE` (VRAM), and `AMP`.
`AMP` is `bf16` for Blackwell / recent GPUs; use `fp16` on older cards (P100, V100)
or `null` for full precision.

In [ ]:
BACKBONE = "efficientnet"   # "efficientnet" or "xception"
HELD_OUT = "FaceSwap"       # DeepFakes | Face2Face | FaceSwap | NeuralTextures
EPOCHS = 15
AMP = "bf16"               # bf16 | fp16 | None

specs = {
    "efficientnet": {"backbone": "efficientnet_b4", "input_size": 224, "batch_size": 64},
    "xception":     {"backbone": "legacy_xception", "input_size": 299, "batch_size": 32},
}
s = specs[BACKBONE]
run_name = f"{BACKBONE}_holdout-{HELD_OUT.lower()}"
cfg = {
    "run_name": run_name, "seed": 1337,
    "backbone": s["backbone"], "pretrained": True, "input_size": s["input_size"],
    "manifest": "data/manifests/crops.parquet",
    "split": f"data/splits/holdout-{HELD_OUT.lower()}.csv",
    "held_out_method": HELD_OUT,
    "epochs": EPOCHS, "batch_size": s["batch_size"], "lr": 0.0003, "amp": AMP,
    "checkpoint_dir": f"checkpoints/{run_name}",
    "video_level": True,
    "extra_test_sets": [{"name": "SimSwap", "split": "data/splits/simswap-test.csv"}],
}
import yaml
cfg_path = f"configs/{run_name}.yaml"
yaml.safe_dump(cfg, open(cfg_path, "w"), sort_keys=False)
print("wrote", cfg_path)
print(open(cfg_path).read())

## 4. Train
Fine-tunes the ImageNet-pretrained backbone on the 3 training methods plus real.
Saves a checkpoint under `checkpoints/<run_name>/`. Uses the GPU if one is visible
(check step 3 of the setup notebook). This is the long cell.

In [ ]:
!python experiments/train.py --config {cfg_path}

## 5. Evaluate
Scores the trained model on each method: the 3 seen methods (in-distribution) and
the held-out method (unseen), plus the SimSwap set if its split exists. Writes
`experiments/results/<run_name>.json` in the shared results schema.

In [ ]:
!python experiments/evaluate.py --config {cfg_path}

## 6. Read the result (the seen-vs-unseen gap)
The headline number is the drop from seen methods to the held-out (unseen) one.

In [ ]:
import json, pandas as pd
r = json.load(open(f"experiments/results/{run_name}.json"))
df = pd.DataFrame(r["results"])
print(f"run: {r['run_name']}  backbone: {r['backbone']}  held-out: {r['held_out_method']}  level: {r['level']}")
print(df[["tested_on", "seen", "auc", "acc", "f1"]].to_string(index=False))
seen = df[df.seen == True]["auc"].mean()
unseen = df[df.seen == False]["auc"].mean()
print(f"\nmean seen AUC:   {seen:.4f}")
print(f"mean unseen AUC: {unseen:.4f}")
print(f"generalization gap (seen - unseen): {seen - unseen:.4f}")

## Next steps
- Run the other folds / backbone (change the two variables in step 3) so all eight
  results JSONs exist, then assemble them into the transfer matrix.
- The held-out method is never opened during training, so its column is a clean
  unseen-generator measurement.
- Optional: log to a shared Weights and Biases project so the runs compare in one
  dashboard (see `docs/INTERFACES.md`, Contract 5).
- The `SimSwap` column stays empty until the self-generated set and its split exist.